# Getting started with Pandora data

This notebook downloads one day of Pandora measurements, plots it, and saves it as
a spreadsheet. You don't need to know anything about the data files to use it.

**How to use it**

1. Run each cell in order, top to bottom: click a cell and press **Shift + Enter**.
2. The only cell you need to change is **Your settings** (step 2). Everything
   else works as is.
3. If a cell shows a red error, read the last line: it usually says what to fix.
   Then fix your settings and run the cells again from step 2.

## Step 1: Install the package

This takes about 30 seconds. On Google Colab you need to run it every time you
open the notebook. You can ignore the messages it prints, including any
suggestion to restart.

In [ ]:
%pip install -q git+https://github.com/paulsavala/pandora-tools.git

## Step 2: Your settings

Change these to look at a different site, instrument, day or product. Keep the
quotation marks around text.

- **LOCATION**: the site's short name, e.g. `"AustinTX"`, `"HoustonTX"`. Step 4
  shows how to find one.
- **INSTRUMENT**: the Pandora number at that site. Step 4 lists them.
- **DATE**: the day to download, written `"YYYY-MM-DD"`. Days are **UTC** days
  (they start at 7 pm the evening before in Texas during daylight saving time).
- **PRODUCT** and **GAS**: which measurement, and which gas in it to look at.
  Pick one row:

  | PRODUCT | GAS | What it is |
  |---|---|---|
  | `"rnvh3"` | `"no2"` | nitrogen dioxide (NO₂), sky scans (also has water vapor, `"h2o"`) |
  | `"rfuh5"` | `"hcho"` | formaldehyde (HCHO), sky scans |
  | `"rfus5"` | `"hcho"` | formaldehyde (HCHO), direct sun |
- **TIMEZONE**: used for the local-time column and the plots.

In [ ]:
LOCATION = "AustinTX"
INSTRUMENT = 257
DATE = "2026-08-28"
PRODUCT = "rnvh3"
GAS = "no2"
TIMEZONE = "America/Chicago"

## Step 3: Connect to the Pandonia Global Network

This loads the package and prepares the connection. Downloaded files are kept in
a folder (shown in the output) so the same day is never downloaded twice.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandora_tools as pdt

client = pdt.PandoraClient(tz=TIMEZONE)
client

## Step 4 (optional): Find a site and instrument

Skip this if you already know your site and instrument number.

Search for sites by part of their name (for example a state abbreviation), or by
distance from a point (latitude, longitude):

In [ ]:
client.find_locations("TX")

In [ ]:
# Sites within 300 km of downtown Austin
client.find_locations(near=(30.27, -97.74), radius_km=300)

Which Pandora instruments are at your site:

In [ ]:
client.instruments(LOCATION)

Which products this instrument has in the 30 days up to your DATE, and on how many days:

In [ ]:
import pandas as pd
start = pd.Timestamp(DATE) - pd.Timedelta(days=30)
client.products(LOCATION, INSTRUMENT, start, DATE)

## Step 5: Download the data

This downloads your day (or reuses it if it was downloaded before) and reads it
into a table called `df`. Each row is one measurement.

In [ ]:
df = client.fetch(LOCATION, INSTRUMENT, DATE, products=PRODUCT)
print(f"{len(df)} measurements")

## Step 6: Look at the data

The table has many columns. The cell below shows the first rows of a few useful ones.
Column names are short codes that are the same in every file:

| Name ending | Meaning |
|---|---|
| `_surf_conc` | concentration at the surface (mol/m³) |
| `_trop_col` | amount in the tropospheric column (mol/m²) |
| `_total_col` | amount in the total column (mol/m²), direct-sun products |
| `_qa` | quality flag (see step 7) |
| `_unc` | uncertainty of the value it follows |

`utc_time` is the measurement time in UTC and `local_time` is the same time in your TIMEZONE.

In [ ]:
df.head()

Not sure what a column means? Search the column descriptions. Replace
`"tropospheric"` with any word, e.g. `"surface"`, `"uncertainty"`, `"angle"`:

In [ ]:
df.find_columns("tropospheric")

Or ask about one column by name (units, full description, special codes):

In [ ]:
df.column_info(f"{GAS}_qa")

## Step 7: Keep only good-quality measurements

Every measurement has a quality flag. `filter_quality` keeps the high and medium
quality ones and drops low-quality and unusable ones. This is almost always what
you want before plotting or averaging.

In [ ]:
good = df.filter_quality(GAS)
print(f"Kept {len(good)} of {len(df)} measurements")

## Step 8: Plot

This plots one column against local time. To plot something else, change
`COLUMN` to any column name from step 6.

In [ ]:
COLUMN = f"{GAS}_trop_col" if f"{GAS}_trop_col" in good else f"{GAS}_total_col"

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(good["local_time"], good[COLUMN], ".", markersize=4)
ax.set_xlabel(f"Local time ({TIMEZONE})")
ax.set_ylabel(f"{COLUMN} ({df.column_info(COLUMN)['units']})")
ax.set_title(f"{COLUMN}, {LOCATION} Pandora {INSTRUMENT}, {DATE} (UTC day)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=good["local_time"].dt.tz))
plt.show()

## Step 9: Save as a spreadsheet

This saves the good-quality measurements as a CSV file, which opens in Excel or
Google Sheets. On Colab, find it in the **Files** panel (folder icon on the left);
right-click it and choose **Download**.

In [ ]:
filename = f"{LOCATION}_{INSTRUMENT}_{PRODUCT}_{DATE}.csv"
good.to_csv(filename, index=False)
print("Saved", filename)

## Next steps

- **Another day or a date range**: `client.fetch(LOCATION, INSTRUMENT, "2026-08-01", "2026-08-07", products=PRODUCT)`
  downloads every day in between (both ends included).
- **Two gases at once**: `no2, hcho = client.fetch(LOCATION, INSTRUMENT, DATE, products=["rnvh3", "rfuh5"])`,
  then `pdt.merge_products(hcho, no2)` lines them up by time.
- **Vertical profiles** (sky-scan products): `good.profiles`.
- **A longer worked example**: `examples/quickstart.ipynb` in the repository.
- **Everything else**: the README on the GitHub page, once you're comfortable.